# Does graph topology really cause the LOPO gain?
## Capacity-matched and shuffled-topology ablation for EV-battery disassembly

The previous experiment showed a strong GNN advantage over an attribute-only MLP on all 10 held-out batteries. This notebook tests whether that gain is caused by **real physical relational topology** rather than simply greater network capacity.

Four neural models are compared under the same 10 leave-one-battery-out folds and 3 fixed seeds:

1. `MLP` — original attribute-only model.
2. `MatchedMLP` — same information as the MLP, automatically widened to closely match the GNN parameter count.
3. `GNN` — original graph model with the true part/fixation topology.
4. `ShuffledGNN` — identical GNN architecture, but the graph topology is deterministically reassigned to nodes while all local operation attributes are preserved.

### Frozen validity checks
- `MatchedMLP` must be within **10%** of the GNN trainable parameter count on every fold.
- `GNN` and `ShuffledGNN` must have identical parameter counts.

### Frozen topology-evidence gate
Continue only if all conditions hold:
1. GNN mean Hits@1 exceeds MatchedMLP by at least **4 percentage points**.
2. GNN mean Hits@1 exceeds ShuffledGNN by at least **4 percentage points**.
3. GNN beats MatchedMLP on at least **7/10** held-out batteries.
4. GNN beats ShuffledGNN on at least **7/10** held-out batteries.
5. GNN mean MRR exceeds both controls.

The task remains reference-sequence ranking, not proof of physical validity of alternative sequences.

In [ ]:
# 1. Imports and frozen configuration
from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32
BASE_SEED = 20261003
GRAPH_SHUFFLE_SEED = 20261119
MODEL_NAMES = ["MLP", "MatchedMLP", "GNN", "ShuffledGNN"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned neural fits:", 10 * len(MODEL_NAMES) * len(SEEDS))

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.

def build_battery_pack(battery_id, vocab, shuffle_graph=False):
    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        degree[a] += 1
        degree[b] += 1

        # Undirected message-passing view of physical fixation relation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    degree_norm = degree / max(1.0, float(degree.max()))

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    # Negative control: reassign the physical topology to node indices while
    # preserving the unlabeled graph structure, edge types, model capacity,
    # and all operation-local attributes.
    graph_degree_norm = degree_norm.copy()
    if shuffle_graph:
        rng_graph = np.random.default_rng(GRAPH_SHUFFLE_SEED + int(battery_id))
        permutation = rng_graph.permutation(n_nodes)
        edge_index = permutation[edge_index]
        shuffled_degree = np.bincount(
            edge_index.reshape(-1), minlength=n_nodes
        ).astype(np.float32)
        graph_degree_norm = shuffled_degree / max(1.0, float(shuffled_degree.max()))

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,  # operation kind: fixation
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,  # removed-part class is not applicable
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            numeric.append([
                degree_norm[src_node],
                degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,  # operation kind: part removal
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": edge_index,
        "edge_type": edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = [], [], [], []
    for t in range(n_ops - 2):
        pool = np.arange(t + 2, n_ops, dtype=np.int64)
        neg = rng.choice(pool, size=k, replace=(len(pool) < k))
        current.append(t); positive.append(t + 1); negative.append(neg)
        progress.append(t / max(1, n_ops - 1))
    return (np.asarray(current,dtype=np.int64), np.asarray(positive,dtype=np.int64),
            np.stack(negative), np.asarray(progress,dtype=np.float32))

def build_model(model_name, vocab, matched_width=None):
    kw = dict(n_skill=len(vocab["skill"]), n_fix=len(vocab["fix"]), n_part=len(vocab["part"]))
    if model_name == "MLP":
        model = BaseOperationRanker(**kw, hidden_dim=HIDDEN_DIM)
    elif model_name == "MatchedMLP":
        assert matched_width is not None
        model = BaseOperationRanker(**kw, hidden_dim=matched_width)
    elif model_name in ["GNN", "ShuffledGNN"]:
        model = GNNOperationRanker(**kw)
    else:
        raise ValueError(model_name)
    return model.to(DEVICE)

def train_one_model(model_name, train_batteries, test_battery, seed, matched_width):
    set_seed(seed)
    vocab = build_fold_vocab(train_batteries)
    shuffled = (model_name == "ShuffledGNN")
    packs = {b: build_battery_pack(b, vocab, shuffle_graph=shuffled)
             for b in train_batteries + [test_battery]}
    model = build_model(model_name, vocab, matched_width)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    for epoch in range(EPOCHS):
        order = list(train_batteries); rng.shuffle(order); model.train()
        for b in order:
            pack = packs[b]
            cur,pos,neg,prog = make_training_pairs(pack["n_ops"], NEGATIVES_PER_STATE, rng)
            z = model.encode_operations(pack)
            cur_t=torch.as_tensor(cur,dtype=torch.long,device=DEVICE)
            pos_t=torch.as_tensor(pos,dtype=torch.long,device=DEVICE)
            neg_t=torch.as_tensor(neg,dtype=torch.long,device=DEVICE)
            prog_t=torch.as_tensor(prog,dtype=torch.float32,device=DEVICE).unsqueeze(-1)
            cur_z=z[cur_t]; pos_z=z[pos_t]
            pos_score=model.score_pairs(cur_z,pos_z,prog_t)
            k=neg_t.shape[1]
            cur_rep=cur_z[:,None,:].expand(-1,k,-1).reshape(-1,cur_z.shape[1])
            neg_z=z[neg_t.reshape(-1)]
            prog_rep=prog_t[:,None,:].expand(-1,k,-1).reshape(-1,1)
            neg_score=model.score_pairs(cur_rep,neg_z,prog_rep).view(len(cur),k)
            loss=F.softplus(-(pos_score[:,None]-neg_score)).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Parameter-count matching helpers

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def instantiate_gnn(vocab):
    return GNNOperationRanker(
        n_skill=len(vocab["skill"]), n_fix=len(vocab["fix"]), n_part=len(vocab["part"])
    )

def choose_matched_mlp_width(vocab, target_parameter_count):
    candidates=[]
    for width in range(HIDDEN_DIM, 257, 2):
        m=BaseOperationRanker(
            n_skill=len(vocab["skill"]), n_fix=len(vocab["fix"]), n_part=len(vocab["part"]),
            hidden_dim=width,
        )
        p=count_trainable_parameters(m)
        candidates.append((abs(p-target_parameter_count),width,p))
    _,width,p=min(candidates,key=lambda x:x[0])
    return width,p

In [ ]:
# 13. Parameter and model smoke tests
TEST_BATTERY = 1
train_batteries = [b for b in sorted(chains) if b != TEST_BATTERY]
vocab = build_fold_vocab(train_batteries)
gnn_probe = instantiate_gnn(vocab)
gnn_params = count_trainable_parameters(gnn_probe)
matched_width, matched_params = choose_matched_mlp_width(vocab, gnn_params)
relative_gap = abs(matched_params-gnn_params)/gnn_params
print("Held-out:", battery_names[TEST_BATTERY])
print("GNN parameters:", gnn_params)
print("MatchedMLP width:", matched_width)
print("MatchedMLP parameters:", matched_params)
print("Relative parameter gap:", f"{100*relative_gap:.2f}%")
assert relative_gap <= 0.10

old_epochs=EPOCHS
EPOCHS=2
for name in ["MatchedMLP","GNN","ShuffledGNN"]:
    model,pack,_=train_one_model(name,train_batteries,TEST_BATTERY,SEEDS[0],matched_width)
    metrics,_=evaluate_neural_model(model,pack,train_batteries)
    print(name, metrics)
EPOCHS=old_epochs
print("Smoke tests passed.")

In [ ]:
# 14. Full 10-fold x 4-model x 3-seed ablation (resumable)
OUT = Path("/content/battery_topology_ablation_lopo")
OUT.mkdir(parents=True, exist_ok=True)
PARTIAL = OUT / "topology_ablation_partial.csv"
results = pd.read_csv(PARTIAL) if PARTIAL.exists() else pd.DataFrame()
if len(results): print("Resuming from", len(results), "completed fits")

battery_ids=sorted(chains)
start_time=time.time()
for fold_idx,test_battery in enumerate(battery_ids,1):
    train_batteries=[b for b in battery_ids if b != test_battery]
    vocab=build_fold_vocab(train_batteries)
    gnn_params=count_trainable_parameters(instantiate_gnn(vocab))
    matched_width,matched_params=choose_matched_mlp_width(vocab,gnn_params)
    parameter_gap=abs(matched_params-gnn_params)/gnn_params
    print(f"\nFold {fold_idx}/10 — held out: {battery_names[test_battery]}")
    print(f"  GNN params={gnn_params:,}; MatchedMLP width={matched_width}; params={matched_params:,}; gap={100*parameter_gap:.2f}%")
    for model_name in MODEL_NAMES:
        for seed in SEEDS:
            done=False
            if len(results):
                done=((results["battery_id"]==test_battery)&(results["model"]==model_name)&(results["seed"]==seed)).any()
            if done:
                print("  skip",model_name,"seed",seed); continue
            model,test_pack,_=train_one_model(model_name,train_batteries,test_battery,seed,matched_width)
            metrics,_=evaluate_neural_model(model,test_pack,train_batteries)
            row={"battery_id":test_battery,"battery":battery_names[test_battery],"model":model_name,
                 "seed":seed,"hits1":metrics["hits1"],"hits3":metrics["hits3"],"mrr":metrics["mrr"],
                 "parameter_count":count_trainable_parameters(model),"gnn_parameter_count":gnn_params,
                 "matched_mlp_width":matched_width,"matched_parameter_gap":parameter_gap}
            results=pd.concat([results,pd.DataFrame([row])],ignore_index=True)
            results.to_csv(PARTIAL,index=False)
            print(f"  {model_name} seed={seed}: H@1={metrics['hits1']:.3f}, H@3={metrics['hits3']:.3f}, MRR={metrics['mrr']:.3f}")
print(f"\nCompleted/resumed to {len(results)} fits in {(time.time()-start_time)/60:.1f} minutes")

In [ ]:
# 15. Fold means and capacity-control validity
fold_mean=(results.groupby(["battery_id","battery","model"],as_index=False)
           .agg(hits1=("hits1","mean"),hits3=("hits3","mean"),mrr=("mrr","mean"),
                parameter_count=("parameter_count","first"),gnn_parameter_count=("gnn_parameter_count","first"),
                matched_parameter_gap=("matched_parameter_gap","first")))
display(fold_mean.sort_values(["battery_id","model"]))
matched_rows=fold_mean[fold_mean["model"]=="MatchedMLP"]
valid_capacity_match=bool((matched_rows["matched_parameter_gap"]<=0.10).all())
gnn_counts=fold_mean[fold_mean["model"]=="GNN"].set_index("battery_id")["parameter_count"]
shuf_counts=fold_mean[fold_mean["model"]=="ShuffledGNN"].set_index("battery_id")["parameter_count"]
valid_identical_gnn_capacity=bool((gnn_counts==shuf_counts).all())
print("MatchedMLP within 10% on every fold:",valid_capacity_match)
print("GNN/ShuffledGNN identical parameter counts:",valid_identical_gnn_capacity)

In [ ]:
# 16. Frozen topology-evidence gate
summary=(fold_mean.groupby("model",as_index=False)
         .agg(mean_hits1=("hits1","mean"),mean_hits3=("hits3","mean"),mean_mrr=("mrr","mean"))
         .sort_values("mean_hits1",ascending=False))
display(summary)
h1=fold_mean.pivot(index="battery",columns="model",values="hits1")
mrr=fold_mean.pivot(index="battery",columns="model",values="mrr")
delta_matched=float((h1["GNN"]-h1["MatchedMLP"]).mean())
delta_shuffled=float((h1["GNN"]-h1["ShuffledGNN"]).mean())
wins_matched=int((h1["GNN"]>h1["MatchedMLP"]).sum())
wins_shuffled=int((h1["GNN"]>h1["ShuffledGNN"]).sum())
gnn_mrr=float(mrr["GNN"].mean()); matched_mrr=float(mrr["MatchedMLP"].mean()); shuffled_mrr=float(mrr["ShuffledGNN"].mean())
V1=valid_capacity_match; V2=valid_identical_gnn_capacity
G1=delta_matched>=0.04; G2=delta_shuffled>=0.04; G3=wins_matched>=7; G4=wins_shuffled>=7
G5=(gnn_mrr>matched_mrr and gnn_mrr>shuffled_mrr)
print("=== TOPOLOGY ABLATION DECISION SUMMARY ===")
print("\n--- Validity ---")
print(f"V1 MatchedMLP within 10% params on all folds: {V1}")
print(f"V2 GNN/ShuffledGNN identical capacity: {V2}")
print("\n--- Topology evidence ---")
print(f"G1 GNN - MatchedMLP Hits@1 >= 4 pp: {100*delta_matched:.2f} pp -> {G1}")
print(f"G2 GNN - ShuffledGNN Hits@1 >= 4 pp: {100*delta_shuffled:.2f} pp -> {G2}")
print(f"G3 GNN beats MatchedMLP on >= 7/10: {wins_matched}/10 -> {G3}")
print(f"G4 GNN beats ShuffledGNN on >= 7/10: {wins_shuffled}/10 -> {G4}")
print(f"G5 GNN MRR > both: GNN={gnn_mrr:.4f}, MatchedMLP={matched_mrr:.4f}, ShuffledGNN={shuffled_mrr:.4f} -> {G5}")
valid_experiment=bool(V1 and V2)
topology_gate=bool(valid_experiment and G1 and G2 and G3 and G4 and G5)
if not valid_experiment: decision="INVALID — FIX CAPACITY CONTROL"
elif topology_gate: decision="TOPOLOGY EFFECT SURVIVES — CONTINUE"
else: decision="TOPOLOGY CLAIM NOT SUPPORTED UNDER FROZEN GATE"
print("\nABLATION DECISION:",decision)

In [ ]:
# 17. Per-battery paired differences
paired=h1.assign(
    real_minus_matched=h1["GNN"]-h1["MatchedMLP"],
    real_minus_shuffled=h1["GNN"]-h1["ShuffledGNN"],
).sort_values("real_minus_shuffled",ascending=False)
display(paired)

In [ ]:
# 18. Save final outputs
FINAL_OUT=Path("/content/battery_topology_ablation_lopo")
results.to_csv(FINAL_OUT/"topology_ablation_seed_results.csv",index=False)
fold_mean.to_csv(FINAL_OUT/"topology_ablation_fold_mean.csv",index=False)
summary.to_csv(FINAL_OUT/"topology_ablation_summary.csv",index=False)
paired.reset_index().to_csv(FINAL_OUT/"topology_ablation_paired.csv",index=False)
record={"valid_capacity_match":valid_capacity_match,"valid_identical_gnn_capacity":valid_identical_gnn_capacity,
        "delta_hits1_gnn_vs_matched":delta_matched,"delta_hits1_gnn_vs_shuffled":delta_shuffled,
        "wins_gnn_vs_matched":wins_matched,"wins_gnn_vs_shuffled":wins_shuffled,
        "mean_mrr_gnn":gnn_mrr,"mean_mrr_matched_mlp":matched_mrr,"mean_mrr_shuffled_gnn":shuffled_mrr,
        "gate_1":G1,"gate_2":G2,"gate_3":G3,"gate_4":G4,"gate_5":G5,"decision":decision}
with open(FINAL_OUT/"topology_ablation_gate.json","w",encoding="utf-8") as f: json.dump(record,f,indent=2)
print("Saved to:",FINAL_OUT)
for p in sorted(FINAL_OUT.iterdir()): print(" -",p.name)

## What to send back

Please send the complete output beginning with:

`=== TOPOLOGY ABLATION DECISION SUMMARY ===`

and preferably upload:

- `topology_ablation_summary.csv`
- `topology_ablation_fold_mean.csv`
- `topology_ablation_paired.csv`

If the gate passes, we will have substantially stronger evidence that the cross-product advantage is tied to **real physical relational structure**, rather than simply additional neural-network capacity.